# Demo 2：参数体系、结果对象与可信审计

本 Notebook 以 Iris 多分类任务为例，集中讲解 `TabularXParams` 的各配置组、`AutoResult` 的阶段结果以及四阶段可信审计对象。它不重复 Demo 1 的基础运行流程。

In [1]:
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import load_iris
from tabularx import AutoEstimator


In [2]:
data = load_iris(as_frame=True)
X, y = data.data, data.target
ae = AutoEstimator(task="classification", output_dir="./outputs/demo2")


## 1. 参数配置器的统一接口

`ae.param` 包含 `profile`、`split`、`preprocess`、`model_search`、`shap` 和 `fairness` 六个配置组。每个配置组都可以使用 `set_param()`、`get_comments()`、`overview()` 和 `to_dict()`；顶层参数对象还可以 `reset()`。

In [3]:
print(ae.param.overview())
print("参数字典：", ae.param.to_dict())
print("split.test_size 说明：", ae.param.split.get_comments("test_size"))


  TabularX 参数总览

【Profile 阶段】
           text_unique_threshold =   200   #文本列判定：唯一值超过此数量 → 可能为文本列
               text_unique_ratio =   0.5   #文本列判定：唯一值占比（唯一值数/行数）超过此值 → 文本列
       int_categorical_threshold =    20   #整数列判定：唯一值数量 ≤ 此值 → 类别列
            scale_mean_threshold =   0.5   #Scale 判据：|mean| > 此值 → 需要标准化
             scale_std_threshold =   1.5   #Scale 判据：std > 此值 → 需要标准化
      high_cardinality_threshold =    50   #高基数判定：唯一值 > 此值 → high_cardinality
                  skew_threshold =   1.0   #偏斜判定：|skew| > 此值 → skewed
                 imbalance_ratio =   0.1   #类别不平衡：最小类别占比 < 此值 → imbalanced
          min_samples_regression =    10   #回归任务最少样本数（低于此值报错）
           min_samples_per_class =     2   #分类任务每类最少样本数（低于此值报错）
            enable_leakage_audit =  True   #是否执行静态数据泄漏与风险预检
         identifier_unique_ratio =  0.98   #疑似标识符特征判定阈值（唯一值比例上限）
          target_proxy_threshold =  0.98   #疑似目标代理特征评分阈值（判别力上限）
                      id_columns =    []   #用户显式指定的 ID 列名列表
              forbi

## 2. ProfileConfig：画像与静态泄露预检

画像阶段记录数据事实，也可以检查高唯一性 ID、目标复制、目标代理和预测时不可用字段。下面展示 API 中的主要配置字段。

In [5]:
ae.param.profile.set_param("identifier_unique_ratio", 0.98)
ae.param.profile.set_param("target_proxy_threshold", 0.98)
ae.param.profile.id_columns = []
ae.param.profile.forbidden_features = []
print(ae.param.profile.to_dict())


{'text_unique_threshold': 200, 'text_unique_ratio': 0.5, 'int_categorical_threshold': 20, 'scale_mean_threshold': 0.5, 'scale_std_threshold': 1.5, 'high_cardinality_threshold': 50, 'skew_threshold': 1.0, 'imbalance_ratio': 0.1, 'min_samples_regression': 10, 'min_samples_per_class': 2, 'enable_leakage_audit': True, 'identifier_unique_ratio': 0.98, 'target_proxy_threshold': 0.98, 'id_columns': [], 'forbidden_features': []}


## 3. SplitConfig：划分策略与完整性审计

`split_strategy` 支持 `auto`、`stratified`、`random`、`group` 和 `temporal`。有实体或时间语义的数据，应显式声明 `group_column` 或 `time_column`。

In [6]:
ae.param.split.set_param("test_size", 0.2)
ae.param.split.set_param("validation_size", 0.25)
ae.param.split.set_param("random_state", 42)
ae.param.split.set_param("split_strategy", "stratified")
ae.param.split.enable_split_audit = True
print(ae.param.split.to_dict())


{'test_size': 0.2, 'validation_size': 0.25, 'random_state': 42, 'min_samples_split': 5, 'split_strategy': 'stratified', 'group_column': None, 'time_column': None, 'enable_split_audit': True, 'duplicate_audit_max_rows': 200000}


## 4. PreprocessConfig：预处理、漂移与审计排除

预处理配置覆盖缺失值填充、缩放、偏斜变换、类别编码、分组列处理、审计建议排除和漂移审计。

In [7]:
ae.param.preprocess.set_param("numeric_impute", "median")
ae.param.preprocess.set_param("categorical_impute", "most_frequent")
ae.param.preprocess.set_param("scale_method", "standard")
ae.param.preprocess.set_param("skew_method", "yeo-johnson")
ae.param.preprocess.set_param("low_cardinality_encoding", "onehot")
ae.param.preprocess.set_param("high_cardinality_encoding", "target")
ae.param.preprocess.drop_group_column = True
ae.param.preprocess.apply_audit_drops = "off"
ae.param.preprocess.enable_drift_audit = True
print(ae.param.preprocess.to_dict())


{'numeric_impute': 'median', 'categorical_impute': 'most_frequent', 'impute_constant_value': 0.0, 'scale_method': 'standard', 'skew_method': 'yeo-johnson', 'low_cardinality_encoding': 'onehot', 'high_cardinality_encoding': 'target', 'max_onehot_categories': 20, 'handle_text': 'drop', 'handle_datetime': 'extract', 'datetime_features': 'basic', 'drop_group_column': True, 'apply_audit_drops': 'off', 'enable_drift_audit': True, 'psi_warning': 0.1, 'psi_high': 0.25, 'ks_warning': 0.1, 'ks_high': 0.2, 'tvd_warning': 0.1, 'tvd_high': 0.25, 'min_group_size': 20, 'drift_bins': 10, 'psi_min_bin_count': 5, 'drift_max_features': 0}


## 5. ModelSearchConfig、ShapConfig 和 FairnessConfig

模型搜索控制候选算法和调参；SHAP 配置控制解释预算；公平性配置控制敏感字段和差异告警阈值。Iris 是多分类任务，本例不请求公平性计算。

In [8]:
ae.param.model_search.models = ["random_forest", "xgboost"]
ae.param.model_search.n_trials = 2
ae.param.model_search.n_jobs = 1
ae.param.model_search.primary_metric = "f1_macro"
ae.param.model_search.device = "cpu"
ae.param.model_search.fixed_params = {}
ae.param.model_search.search_space_overrides = {}
ae.param.shap.max_explain_samples = 24
ae.param.shap.n_background = 20
ae.param.shap.top_k_features = 4
ae.param.fairness.sensitive_features = []
ae.param.fairness.dp_threshold = 0.1
ae.param.fairness.eo_tpr_threshold = 0.1
ae.param.fairness.eo_fpr_threshold = 0.1


## 6. 运行并查看 AutoResult

`AutoResult` 将各阶段结果挂载在一个对象上：`profile`、`split`、`preprocessed`、`searched`、`shap`、`fairness`、`output_dir` 和 `report_paths`。

In [9]:
ae.fit(X, y)
r = ae.result
print("profile 行数/列数：", r.profile.n_rows, r.profile.n_cols)
print("train/valid/test：", len(r.split.train_indices), len(r.split.valid_indices), len(r.split.test_indices))
print("预处理训练集：", r.preprocessed.X_train.shape)
print("最佳模型：", r.searched.best_model_name)
print("候选模型：", list(r.searched.model_runs.keys()) if hasattr(r.searched, "model_runs") else r.searched)


[I 2026-10-07 22:22:08,824] A new study created in memory with name: random_forest_multiclass


profile 行数/列数： 150 4
train/valid/test： 90 30 30
预处理训练集： (90, 4)
最佳模型： xgboost
候选模型： ModelSearchResult(task_type='multiclass', primary_metric='f1_macro', optimization_direction='maximize', model_results=[ModelRunResult(model_name='random_forest', task_type='multiclass', status='success', best_params={'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'ccp_alpha': 0.04330880728874676}, n_trials=2, optuna_duration_seconds=0.8, optuna_trials=[TrialRecord(trial_number=0, params={'n_estimators': 200, 'max_depth': 30, 'min_samples_split': 15, 'min_samples_leaf': 6, 'max_features': 'sqrt', 'ccp_alpha': 0.04330880728874676}, value=0.899749373433584, state='COMPLETE', duration_seconds=0.249126), TrialRecord(trial_number=1, params={'n_estimators': 350, 'max_depth': 24, 'min_samples_split': 2, 'min_samples_leaf': 10, 'max_features': 'sqrt', 'ccp_alpha': 0.00917022549267169}, value=0.899749373433584, state='COMPLETE', duration_seconds=0.546

## 7. 查看四阶段审计结果

画像、划分、预处理和模型搜索都会产生审计结果。审计的意义是暴露风险和记录检查状态，不是替用户做业务判断。

In [10]:
audit_objects = {
    "profile": r.profile.audit,
    "split": r.split.audit,
    "preprocess": r.preprocessed.audit,
}
for name, audit in audit_objects.items():
    print(name, "风险等级：", audit.risk_level)
    print(audit.to_dict())

print("模型搜索审计文件：", r.report_paths.get("model_search"))


profile 风险等级： info
{'risk_level': 'info', 'high_risk_count': 0, 'findings': [], 'check_results': [{'code': 'target_copy', 'check': '目标复制检测', 'status': '通过', 'result': '未发现特征与目标逐行一致。'}, {'code': 'possible_target_proxy', 'check': '目标代理特征检测', 'status': '通过', 'result': '未发现超过阈值的疑似目标代理特征。'}, {'code': 'identifier_like_column', 'check': 'ID/高唯一性字段检测', 'status': '通过', 'result': '未发现疑似 ID 或超高唯一性字段。'}, {'code': 'forbidden_feature', 'check': '用户禁用特征校验', 'status': '通过', 'result': '未发现包含声明的非法特征。'}], 'model_info_questions': ['每个保留特征是否在真实预测时已经可获得，而不是结果发生后才产生？', '测试集是否代表要泛化到的人群、地点与时间段？', '是否存在实体、时间、空间或层级依赖，需要相应的分组切分？'], 'suggested_drop_columns': []}
split 风险等级： critical
{'risk_level': 'critical', 'high_risk_count': 1, 'findings': [{'code': 'cross_split_duplicates', 'severity': 'critical', 'title': '跨集合重复样本', 'evidence': '发现 1 个原始特征行同时出现在多个集合中。', 'recommendation': '在切分前去重，或让重复实体只进入一个集合，否则测试分数会被高估。', 'columns': [], 'details': {'count': 1, 'sampled': False}}], 'check_results': [{'code': 'cross_split_dupl

## 本例小结

本例覆盖全部参数配置组、结果契约入口和阶段审计结果。若要了解文件级交付、失败恢复和异常捕获，请继续阅读 Demo 3。